[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/supplemental/A1_Why_Is_It_Blank.ipynb)

# A1: Why Is It Blank?
**CS/DS 225 · companion to 08.2 Missing Data · supplemental activity**

Keep the **A1 assignment in Canvas** open next to you: your partner types your answers there while you run the cells. **Run the cells in order.** Each cell is labeled with the assignment question it answers. For Questions 6-9 and 10 you do not need to change any code; Question 13 asks you to edit one line.

Write your team's answers in the **Canvas assignment**, not here. If you want to try an extra idea, add a new cell at the bottom.

## Cell 1: Setup (just run it)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# the Fun Survey responses (live sheet: counts may differ by a few from the printed ones)
url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRkK73xD192AdP0jZe6ac9cnVPSeqqbYZmSPnhY2hnY8ANROAOCStRFdvjwFoapv3j2rzMtZ91KXPFm/pub?output=csv"
df = pd.read_csv(url)

df.columns = ['Timestamp', 'musicartist', 'height', 'city', '30min', 'travel',
              'likepizza', 'deepdish', 'sport', 'spell', 'hangout', 'talk',
              'year', 'quote', 'areacode', 'pets', 'superpower', 'shoes']

# turn the text timestamps into real dates and times
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%m/%d/%Y %H:%M:%S')

# the four questions with lots of blanks
cols = ['areacode', 'pets', 'superpower', 'shoes']

print("rows:", len(df))
print("first response:", df['Timestamp'].min())
print("last response: ", df['Timestamp'].max())
df[['Timestamp'] + cols].head()

## Cell 2: Question 6. How many blanks in each column?

In [ ]:
df[cols].isnull().sum()

## Cell 3: Question 7. Blank in all four at once?
`all(axis=1)` asks, for each **row**, whether every one of the four columns is blank. `.sum()` counts the rows where the answer is yes.

In [ ]:
df[cols].isnull().all(axis=1).sum()

## Cell 4: Question 8. Percent blank, year by year
Compare with the guess you wrote for Question 5.

In [ ]:
df['year_taken'] = df['Timestamp'].dt.year

# percent of rows blank in each column, for each year
pct_blank = df.groupby('year_taken')[cols].apply(lambda g: (g.isnull().mean() * 100).round(0))
pct_blank['rows that year'] = df.groupby('year_taken').size()
pct_blank

## Cell 5: Question 9. When was each question first answered?

In [ ]:
for c in cols:
    first = df.loc[df[c].notna(), 'Timestamp'].min()
    print(f"{c:12} first answered: {first}")

# the last row before the change and the first row after it
# (the sheet is in time order; rows 203 and 204 are the ones from Model 1)
df.loc[203:204, cols].notna().join(df['Timestamp'])   # True = answered, False = blank

## Cell 6: Question 10. Two ways to drop rows

In [ ]:
print("rows in the full sheet:                  ", len(df))
print("dropna(subset=cols) keeps:               ", len(df.dropna(subset=cols)))
print("dropna(subset=cols, how='all') keeps:    ", len(df.dropna(subset=cols, how='all')))

## Cell 7: Question 13. Keep only the rows where the questions existed
**Edit the date on the line marked EDIT** so it is the date your team found in Question 9, then run the cell.

In [ ]:
cutoff = pd.Timestamp('2018-01-01')    # EDIT: replace with the first-answered date from Question 9

after = df[df['Timestamp'] >= cutoff]

print("rows kept:", len(after))
after[cols].isnull().sum()

## Cell 8 (optional): see the blanks
Each column of the picture is one response, in time order (earliest on the left). Yellow means blank. Does the picture match what you found in Questions 7-9?

In [ ]:
plt.figure(figsize=(12, 2.5))
sns.heatmap(df[cols].isnull().T, cbar=False, cmap='viridis', yticklabels=cols, xticklabels=False)
plt.title("Blank answers across all responses (yellow = blank, time runs left to right)")
plt.xlabel("Responses, earliest to latest")
plt.show()